# Rate arithmetic

A flow rate is an expression tree. `tanh`, `abs`, `clip` and `**` are nodes
in that tree, so a triangular seed or a detection scale-up does not have to
hide inside a `Transform` callback. The same operators apply to a saved
`Trace`. A parameter-only piece such as `tanh(Param("se"))` is evaluated
with `eval_closed` and then scales the trace.


## Triangular seed

tb_macro's seeding pulse is `clip(h * (1 - abs(t - peak) / width), 0)`:
a triangle of height `h` and width `width`, zero outside that window.


In [ ]:
import numpy as np

from summer4 import (
    FlowModel,
    Param,
    Property,
    PropertyMap,
    Time,
    TransitionFlow,
    clip,
)

state = Property("state", ("S", "I"))
pmap = PropertyMap.from_property(state)
seed = clip(Param("height") * (1 - abs(Time() - Param("peak")) / Param("width")), 0)
model = FlowModel(pmap)
model.add_flow(TransitionFlow("seed", state["S"], state["I"], seed, absolute=True))
compiled = model.compile()
params = {"height": 0.05, "peak": 20.0, "width": 8.0}
times = np.linspace(0.0, 40.0, 50)
y = np.array([1.0, 0.0])
got = np.array(
    [float(np.asarray(compiled.vector_field(float(t), y, params))[1]) for t in times]
)
expect = np.maximum(
    params["height"] * (1.0 - np.abs(times - params["peak"]) / params["width"]),
    0.0,
)
np.testing.assert_allclose(got, expect, rtol=1e-5, atol=1e-6)
assert got[0] == 0.0 and got[-1] == 0.0
assert got.max() > 0.9 * params["height"]
print(f"peak sample {got.max():.4f} (height {params['height']})")


## Tanh scale-up

Detection rises from `start` to `end`, with the midpoint at `inflection` and
steepness `shape`:
`start + (end - start) * (tanh(shape * (t - inflection)) + 1) / 2`.


In [ ]:
from summer4 import tanh

scale = Param("start") + (Param("end") - Param("start")) * (
    tanh(Param("shape") * (Time() - Param("inflection"))) + 1
) / 2
detect = FlowModel(pmap)
detect.add_flow(
    TransitionFlow("detect", state["S"], state["I"], scale, absolute=True)
)
compiled = detect.compile()
params = {"shape": 0.3, "inflection": 18.0, "start": 0.1, "end": 1.0}
got = np.array(
    [float(np.asarray(compiled.vector_field(float(t), y, params))[1]) for t in times]
)
expect = params["start"] + (params["end"] - params["start"]) * (
    np.tanh(params["shape"] * (times - params["inflection"])) + 1.0
) / 2.0
np.testing.assert_allclose(got, expect, rtol=1e-5, atol=1e-6)
assert got[0] < got[-1]
print(f"scale-up {got[0]:.3f} -> {got[-1]:.3f}")


## The same transform on a trace

`tanh(Param("se"))` is a rate expression. `eval_closed` evaluates the
parameter-only part against a parameter dict. The array scales a `Trace`
with the same operator the rate tree would use. The trace does not carry
parameters, so multiplying by the unevaluated expression is an error.


In [ ]:
from summer4 import Trace, eval_closed
from summer4.time import TimeAxis

expr = tanh(Param("se"))
factor = float(eval_closed(expr, {"se": 0.4}))
np.testing.assert_allclose(factor, np.tanh(0.4), rtol=1e-5, atol=1e-6)

constant = FlowModel(pmap)
constant.add_flow(
    TransitionFlow("detect", state["S"], state["I"], expr, absolute=True)
)
rate = float(np.asarray(constant.compile().vector_field(0.0, y, {"se": 0.4}))[1])
np.testing.assert_allclose(rate, factor, rtol=1e-5, atol=1e-6)

values = np.array([10.0, 20.0, 30.0, 40.0])
trace = Trace(
    times=TimeAxis(values=np.linspace(0.0, 3.0, 4), epoch=None, kind="explicit"),
    values=values,
    dims=("time",),
)
scaled = trace * factor
np.testing.assert_allclose(np.asarray(scaled.values), values * factor, rtol=1e-5)
try:
    trace * expr
except TypeError as exc:
    assert "eval_closed" in str(exc)
else:
    raise AssertionError("trace * unevaluated expr should fail")
print(f"sensitivity {factor:.4f}, scaled incidence {np.asarray(scaled.values)}")


## Per-age death rates

`Interp` is one series. A death-rate table is one series per age band, and
writing each band out as its own tree is a thousand knot nodes. `Data.table`
holds the whole grid, and `interp` evaluates every column in one node. The
result is a `GroupedRate` over the age property, so an exit flow applies each
band's rate to that band.

In [ ]:
import pandas as pd

from summer4 import ExitFlow, PropertyData
from summer4.data import Data

age = Property("age", ("0", "15", "65"))
alive = Property("state", ("Y",))
death_times = np.array([2000.0, 2010.0, 2020.0])
deaths = pd.DataFrame(
    {
        "0": [0.020, 0.015, 0.010],
        "15": [0.004, 0.005, 0.006],
        "65": [0.040, 0.045, 0.050],
    }
)
table = Data.table(death_times, deaths, over=age)
age_map = PropertyMap.from_property(alive).stratify(age)
deaths_model = FlowModel(age_map)
deaths_model.add_flow(ExitFlow("die", alive["Y"], table.interp()))
compiled_deaths = deaths_model.compile()
pop = PropertyData.wrap(age_map, np.ones(age_map.size))
# Halfway from 2000 to 2010, one person in each band.
mid = np.asarray(compiled_deaths.vector_field(2005.0, pop, {}).data)
expect = -0.5 * (deaths.iloc[0].to_numpy() + deaths.iloc[1].to_numpy())
np.testing.assert_allclose(mid, expect, rtol=1e-5, atol=1e-6)
after = np.asarray(compiled_deaths.vector_field(2030.0, pop, {}).data)
np.testing.assert_allclose(after, -deaths.iloc[-1].to_numpy(), rtol=1e-5, atol=1e-6)
print(f"death rates at 2005: {-mid}")

## A mixing matrix per year

The contact matrix changes by year, but it does not have to be rebuilt inside
the vector field. The stack of yearly matrices is a parameter. `Lookup`
gathers the row for `floor(Time() - 1850)`, and that matrix is the mixing
matrix. A year before the first row holds the first matrix.

In [ ]:
from summer4 import floor
from summer4.epi import ForceOfInfection, MixingMatrix
from summer4.flows.rates import Lookup

stack = np.stack(
    [
        np.array([[0.8, 0.2], [0.3, 0.7]]),
        np.array([[0.5, 0.5], [0.5, 0.5]]),
        np.array([[0.1, 0.9], [0.4, 0.6]]),
    ]
)
age2 = Property("age", ("young", "old"))
state2 = Property("state", ("S", "I", "R"))
pmap2 = PropertyMap.from_property(state2).stratify(age2)
foi = ForceOfInfection(
    "infection",
    infectious=state2["I"],
    group_by=age2,
    kind="frequency",
    contact_rate=0.4,
    mixing=MixingMatrix(
        age2,
        Lookup(Param("stack"), floor(Time() - 1850.0)),
        normalize="none",
        check_reciprocal=False,
    ),
)
mix_model = FlowModel(pmap2)
mix_model.add_flow(TransitionFlow("inf", state2["S"], state2["I"], foi))
compiled_mix = mix_model.compile()
y2 = np.array([900.0, 800.0, 80.0, 10.0, 0.0, 0.0])
# 1851.2 is year index 1, whose rows are identical, so both ages share one FOI.
foi_mid = np.asarray(
    compiled_mix.observe(1851.2, y2, {"stack": stack}).captures["infection"].data
)
np.testing.assert_allclose(foi_mid[0], foi_mid[1], rtol=1e-5, atol=1e-6)
# 1840 clamps to the first, assortative, matrix.
foi_early = np.asarray(
    compiled_mix.observe(1840.0, y2, {"stack": stack}).captures["infection"].data
)
assert float(np.max(np.abs(foi_early[0] - foi_early[1]))) > 0.0
print(f"FOI in 1851 {foi_mid}, FOI in 1840 {foi_early}")